# 03. Correlation-based feature selection + direct PCA embeddings

Replaces the old group-wise PCA. New behavior:

1. Rank every numeric feature by absolute Pearson correlation with the
   psychosis target (case = psychosis_risk, control = non_psychosis by default).
2. Take the top 10 / 20 / 30 features (nested) and build **one direct PCA
   embedding per cut**. No feature groups, no `omop_price` mapping file.
3. Save a correlation Excel and three embedding parquet files.

Outputs (in `output/embeddings/`):
- `feature_correlations.xlsx`   (selected features + Pearson r; full ranking on a second sheet)
- `embedding_top10.parquet`, `embedding_top20.parquet`, `embedding_top30.parquet`  (row_id + PCA scores)
- `embeddings_metadata.csv`     (explained variance per component)
- `embedding_selected_features.csv`  (which features feed each embedding)

The case/control label table is produced separately in notebook 04.


## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

INPUT_DIR = Path('output')
OUTPUT_DIR = INPUT_DIR / 'embeddings'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_MATRIX_PATH = INPUT_DIR / '07_feature_matrix.parquet'
DISEASE_MATRIX_PATH = INPUT_DIR / '08_disease_matrix.parquet'

ROW_ID = 'row_id'
PERSON_ID = 'person_id'
INDEX_TIME = 'index_datetime'
COHORT_GROUP_COL = 'cohort_group'

# --- Correlation target -------------------------------------------------------
# Features are ranked by |Pearson r| against this binary target.
# 'cohort' : case = psychosis_risk vs control = non_psychosis (the psychosis-disease group).
# 'phecode': a specific PheCode column in 08_disease_matrix.parquet.
CORR_TARGET_MODE = 'cohort'              # 'cohort' or 'phecode'
POSITIVE_COHORT_LABEL = 'psychosis_risk' # positive class when mode == 'cohort'
PHECODE_TARGET_COL = None                # e.g. '295__Schizophrenia...' when mode == 'phecode'

# --- Embedding configuration --------------------------------------------------
TOP_N_LIST = [10, 20, 30]   # top-correlated features used per embedding
N_COMPONENTS = None         # PCA components per embedding; None = keep all (= n input features)
MIN_OBS_FOR_CORR = 50       # a feature needs at least this many observed values to be ranked

print('Feature matrix:', FEATURE_MATRIX_PATH)
print('Output dir     :', OUTPUT_DIR.resolve())
print('Target mode    :', CORR_TARGET_MODE)
print('Top-N list     :', TOP_N_LIST)
print('N_COMPONENTS   :', N_COMPONENTS)


## 2. Load feature matrix and build the correlation target

In [ ]:
if not FEATURE_MATRIX_PATH.exists():
    raise FileNotFoundError(FEATURE_MATRIX_PATH)

X_raw = pd.read_parquet(FEATURE_MATRIX_PATH)
print('Feature matrix:', X_raw.shape)
print('Cohort distribution:')
print(X_raw[COHORT_GROUP_COL].value_counts())

# Identifier columns are never treated as features.
identifier_cols = {ROW_ID, PERSON_ID, INDEX_TIME, COHORT_GROUP_COL}

# Candidate features: numeric columns only (age_at_index + measurement values).
# Categorical demographics are non-numeric strings, so they are excluded here.
candidate_features = [
    c for c in X_raw.columns
    if c not in identifier_cols and pd.api.types.is_numeric_dtype(X_raw[c])
]
print('Candidate numeric features:', len(candidate_features))

if CORR_TARGET_MODE == 'cohort':
    target = (X_raw[COHORT_GROUP_COL] == POSITIVE_COHORT_LABEL).astype('float64').to_numpy()
    target_name = f'cohort=={POSITIVE_COHORT_LABEL}'
elif CORR_TARGET_MODE == 'phecode':
    if PHECODE_TARGET_COL is None:
        raise ValueError('Set PHECODE_TARGET_COL when CORR_TARGET_MODE == "phecode".')
    Y_raw = pd.read_parquet(DISEASE_MATRIX_PATH)
    Y_aligned = X_raw[[ROW_ID]].merge(Y_raw, on=ROW_ID, how='left')
    if PHECODE_TARGET_COL not in Y_aligned.columns:
        raise ValueError(f'{PHECODE_TARGET_COL} not in disease matrix columns.')
    target = pd.to_numeric(Y_aligned[PHECODE_TARGET_COL], errors='coerce').fillna(0).to_numpy()
    target_name = PHECODE_TARGET_COL
else:
    raise ValueError("CORR_TARGET_MODE must be 'cohort' or 'phecode'.")

print('Target        :', target_name)
print('Positive rate :', float(np.mean(target)))


## 3. Pearson correlation per feature (observed values only)

In [ ]:
# Use observed (non-missing) values per feature so missingness does not dilute r.
rows = []
for c in candidate_features:
    v = pd.to_numeric(X_raw[c], errors='coerce').to_numpy()
    obs = ~np.isnan(v)
    n_obs = int(obs.sum())
    if n_obs < MIN_OBS_FOR_CORR:
        r = np.nan
    else:
        vv, tt = v[obs], target[obs]
        if np.std(vv) == 0 or np.std(tt) == 0:
            r = np.nan
        else:
            r = float(np.corrcoef(vv, tt)[0, 1])
    rows.append({'feature': c, 'pearson_r': r,
                 'abs_r': abs(r) if r == r else np.nan,
                 'n_observed': n_obs, 'observed_fraction': n_obs / len(v)})

corr_df = (pd.DataFrame(rows)
           .sort_values('abs_r', ascending=False, na_position='last')
           .reset_index(drop=True))
corr_df['rank'] = np.arange(1, len(corr_df) + 1)

ranked = corr_df.dropna(subset=['pearson_r']).reset_index(drop=True)
if len(ranked) < max(TOP_N_LIST):
    raise ValueError(f'Only {len(ranked)} features have a valid correlation, '
                     f'fewer than max(TOP_N_LIST)={max(TOP_N_LIST)}.')

# Nested selection: top10 is the first 10 of the ranking, top20 the first 20, etc.
selected = {n: ranked.head(n)['feature'].tolist() for n in TOP_N_LIST}
for n in TOP_N_LIST:
    corr_df[f'selected_top{n}'] = corr_df['feature'].isin(selected[n])

print('Top 10 features by |Pearson r| with', target_name)
display(ranked.head(10)[['rank', 'feature', 'pearson_r', 'n_observed', 'observed_fraction']])


## 4. Save correlation Excel

In [ ]:
CORR_XLSX_PATH = OUTPUT_DIR / 'feature_correlations.xlsx'

top_block = ranked.head(max(TOP_N_LIST)).copy()
for n in TOP_N_LIST:
    top_block[f'selected_top{n}'] = top_block['feature'].isin(selected[n])
top_block = top_block[['rank', 'feature', 'pearson_r', 'abs_r',
                       'n_observed', 'observed_fraction'] +
                      [f'selected_top{n}' for n in TOP_N_LIST]]

with pd.ExcelWriter(CORR_XLSX_PATH, engine='openpyxl') as xw:
    top_block.to_excel(xw, sheet_name=f'top{max(TOP_N_LIST)}_selected', index=False)
    corr_df.to_excel(xw, sheet_name='full_ranking', index=False)

print('Saved:', CORR_XLSX_PATH)
display(top_block.head(30))


## 5. Standardize selected features (observed -> z, missing -> 0)

In [ ]:
# Standardize observed values per feature, then zero-fill missing.
# Standardization is scale-only (does not change r) and puts lab values on a
# common scale before PCA. top30 is a superset of top20/top10, so standardize once.
union_features = selected[max(TOP_N_LIST)]
z_cols = {}
for c in union_features:
    v = pd.to_numeric(X_raw[c], errors='coerce')
    mean, sd = v.mean(), v.std(ddof=0)
    z = (v - mean) if (not np.isfinite(sd) or sd == 0) else (v - mean) / sd
    z_cols[c] = z.astype('float32')

Z_filled = pd.DataFrame(z_cols).fillna(0.0).astype('float32')
print('Standardized feature block:', Z_filled.shape,
      '| missing after zero-fill:', int(Z_filled.isna().sum().sum()))


## 6. Direct PCA embedding for each top-N cut

In [ ]:
embedding_meta_rows = []

for n in TOP_N_LIST:
    feats = selected[n]
    M = Z_filled[feats].to_numpy(dtype=np.float32, copy=True)
    k = M.shape[1] if N_COMPONENTS is None else min(N_COMPONENTS, M.shape[1])

    pca = PCA(n_components=k, svd_solver='full')
    scores = pca.fit_transform(M).astype('float32')
    pc_names = [f'PC{i:03d}' for i in range(1, k + 1)]

    emb = pd.DataFrame(scores, columns=pc_names)
    emb.insert(0, ROW_ID, X_raw[ROW_ID].to_numpy())

    out_path = OUTPUT_DIR / f'embedding_top{n}.parquet'
    emb.to_parquet(out_path, index=False)

    total_var = float(np.sum(pca.explained_variance_ratio_))
    print(f'[top{n}] {len(feats)} features -> {k} PCs '
          f'| total explained variance = {total_var:.4f} -> {out_path.name}')

    for i, name in enumerate(pc_names):
        embedding_meta_rows.append({
            'embedding': f'top{n}', 'n_input_features': len(feats),
            'component': name,
            'explained_variance_ratio': float(pca.explained_variance_ratio_[i]),
        })

# Long-form record of which features feed each embedding, with their r.
sel_rows = []
for n in TOP_N_LIST:
    for rk, feat in enumerate(selected[n], start=1):
        r = float(ranked.loc[ranked['feature'] == feat, 'pearson_r'].iloc[0])
        sel_rows.append({'embedding': f'top{n}', 'rank': rk, 'feature': feat, 'pearson_r': r})

EMB_META_PATH = OUTPUT_DIR / 'embeddings_metadata.csv'
SEL_FEAT_PATH = OUTPUT_DIR / 'embedding_selected_features.csv'
pd.DataFrame(embedding_meta_rows).to_csv(EMB_META_PATH, index=False)
pd.DataFrame(sel_rows).to_csv(SEL_FEAT_PATH, index=False)
print('Saved:', EMB_META_PATH)
print('Saved:', SEL_FEAT_PATH)


## 7. Output manifest

In [ ]:
print('Embedding outputs in', OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.glob('*')):
    if p.is_file():
        print(f'  {p.name:42s} {p.stat().st_size / 1024**2:8.2f} MB')
